# 07 — Explicabilidad comparada v0

Versión inicial del notebook comparativo.

Esta versión **no depende todavía del dataset MIMIC-CXR ni de checkpoints fine-tuneados**. Usa una imagen local y el modelo `blip_base`, pero deja armada la estructura final de outputs para escalar después a:

```text
25 radiografías × 3 modelos × 3 métodos
```

## Objetivo de esta v0

Validar dentro del layout final:

```text
outputs/notebook_comparativo/
├── captions/
├── heatmaps/
├── arrays/
├── metrics/
├── figures_paper/
└── summary.csv
```

Flujo:

```text
imagen local
→ BLIP base
→ caption best-of-N
→ generated_ids
→ post-softmax cross-attention
→ QK logits
→ Grad-CAM
→ métricas espaciales
→ figuras
→ summary.csv
```


## 0. Uso

Guardar este notebook en:

```text
image-captioning/notebooks/07_explicabilidad_comparada_v0.ipynb
```

Abrir Jupyter desde la raíz del repo:

```bash
cd ~/Documents/Vision\ Artificial/tp_final_vision/image-captioning
jupyter notebook
```

Luego abrir este archivo desde `notebooks/` y ejecutar las celdas en orden.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, Markdown

def find_repo_root(start: Path | None = None) -> Path:
    """Busca la raíz del repo subiendo desde el cwd hasta encontrar src/ y models/."""
    start = (start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "src").exists() and (p / "models").exists():
            return p
    raise RuntimeError(
        "No pude encontrar la raíz del repo. "
        "Abrí Jupyter desde image-captioning/ o ajustá ROOT manualmente."
    )

ROOT = find_repo_root()
print("ROOT:", ROOT)


## 1. Configuración

Modo actual:

```text
RUN_MODE = "local_image"
```

Esto permite avanzar aunque todavía no esté disponible el cache local de MIMIC-CXR.

Cuando tengas dataset y checkpoints, esta misma estructura se puede escalar cambiando `ITEMS` y `MODELS`.


In [ ]:
RUN_MODE = "local_image"

# Output final del notebook comparativo.
OUT = ROOT / "outputs/notebook_comparativo"

# Imagen local ya validada.
ITEMS = [
    {
        "idx": "local_prueba1",
        "image_path": ROOT / "data/img_prueba/prueba1.jpeg",
        "reference": "Imagen local de prueba; no corresponde a una radiografía MIMIC-CXR.",
    }
]

# Por ahora solo está disponible BLIP base localmente.
MODELS = {
    "base": ROOT / "models/blip_base",
}

DEVICE = "cpu"
SEEDS = [42]
MAX_NEW_TOKENS = 12
SKIP_GRADCAM = False

FORCE_RUN = False
FORCE_PLOT = False

print("RUN_MODE:", RUN_MODE)
print("OUT:", OUT)
print("DEVICE:", DEVICE)
print("MODELS:")
for tag, path in MODELS.items():
    print(f"  {tag}: {path}")

print("ITEMS:")
for item in ITEMS:
    print(f"  {item['idx']}: {item['image_path']}")


## 2. Crear estructura de outputs

Esta celda crea el layout final esperado para el notebook comparativo.


In [ ]:
DIRS = {
    "captions": OUT / "captions",
    "heatmaps": OUT / "heatmaps",
    "arrays": OUT / "arrays",
    "metrics": OUT / "metrics",
    "figures_paper": OUT / "figures_paper",
    "work": OUT / "_work_single_image",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Directorios creados:")
for name, d in DIRS.items():
    print(f"  {name}: {d.relative_to(ROOT)}")


## 3. Validar disponibilidad de archivos

Esta celda no carga el modelo todavía. Solo confirma que existen las rutas necesarias.


In [ ]:
assert (ROOT / "scripts/run_single_image_compare.py").exists(), "Falta scripts/run_single_image_compare.py"
assert (ROOT / "scripts/plot_single_image_compare.py").exists(), "Falta scripts/plot_single_image_compare.py"

for tag, model_dir in MODELS.items():
    assert model_dir.exists(), f"No existe modelo {tag}: {model_dir}"
    assert (model_dir / "config.json").exists(), f"Falta config.json en {model_dir}"
    assert (model_dir / "model.safetensors").exists() or (model_dir / "pytorch_model.bin").exists(), (
        f"Falta model.safetensors o pytorch_model.bin en {model_dir}"
    )

for item in ITEMS:
    assert item["image_path"].exists(), f"No existe imagen: {item['image_path']}"

print("Validación OK.")


## 4. Mostrar imágenes de entrada

In [ ]:
for item in ITEMS:
    img = Image.open(item["image_path"]).convert("RGB")
    print(item["idx"], img.size, img.mode)
    display_width = 360
    display_height = int(display_width * img.size[1] / img.size[0])
    display(Markdown(f"### {item['idx']}"))
    display(img.resize((display_width, display_height)))


## 5. Ejecutar extracción por imagen/modelo

Esta celda llama a:

```text
scripts/run_single_image_compare.py
```

y guarda resultados temporales en:

```text
outputs/notebook_comparativo/_work_single_image/<idx>/<model_tag>/
```

Luego otras celdas reorganizan esos outputs al layout final.


In [ ]:
def run_command(cmd: list[str], cwd: Path) -> None:
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

run_records = []

for item in ITEMS:
    idx = item["idx"]
    image_path = item["image_path"]

    for model_tag, model_dir in MODELS.items():
        work_dir = DIRS["work"] / idx / model_tag
        caption_path = work_dir / "caption.json"
        npz_path = work_dir / "compare_heatmaps.npz"
        metrics_csv = work_dir / "spatial_metrics_rows.csv"

        should_run = FORCE_RUN or not (caption_path.exists() and npz_path.exists() and metrics_csv.exists())

        if should_run:
            work_dir.mkdir(parents=True, exist_ok=True)

            cmd = [
                sys.executable,
                str(ROOT / "scripts/run_single_image_compare.py"),
                "--image-path", str(image_path),
                "--model-dir", str(model_dir),
                "--output-dir", str(work_dir),
                "--device", DEVICE,
                "--seeds", *map(str, SEEDS),
                "--max-new-tokens", str(MAX_NEW_TOKENS),
            ]

            if SKIP_GRADCAM:
                cmd.append("--skip-gradcam")

            print(f"\n=== Running {idx} / {model_tag} ===")
            run_command(cmd, cwd=ROOT)
        else:
            print(f"Cache existente, salteando: {idx} / {model_tag}")

        run_records.append({
            "idx": idx,
            "model_tag": model_tag,
            "model_path": str(model_dir.relative_to(ROOT)),
            "image_path": str(image_path.relative_to(ROOT)),
            "work_dir": str(work_dir.relative_to(ROOT)),
            "caption_path": str(caption_path.relative_to(ROOT)),
            "npz_path": str(npz_path.relative_to(ROOT)),
            "metrics_csv": str(metrics_csv.relative_to(ROOT)),
        })

run_records


## 6. Generar figuras por imagen/modelo

Esta celda llama a:

```text
scripts/plot_single_image_compare.py
```

y guarda `explanation.png` dentro del layout final:

```text
outputs/notebook_comparativo/heatmaps/<idx>/<model_tag>/explanation.png
```


In [ ]:
figure_records = []

for rec in run_records:
    idx = rec["idx"]
    model_tag = rec["model_tag"]
    image_path = ROOT / rec["image_path"]
    npz_path = ROOT / rec["npz_path"]

    final_heatmap_dir = DIRS["heatmaps"] / idx / model_tag
    final_heatmap_dir.mkdir(parents=True, exist_ok=True)

    original_path = final_heatmap_dir / "original.png"
    explanation_path = final_heatmap_dir / "explanation.png"

    if FORCE_PLOT or not original_path.exists():
        Image.open(image_path).convert("RGB").save(original_path)

    if FORCE_PLOT or not explanation_path.exists():
        tmp_fig_dir = final_heatmap_dir / "_tmp_figures"
        tmp_fig_dir.mkdir(parents=True, exist_ok=True)

        cmd = [
            sys.executable,
            str(ROOT / "scripts/plot_single_image_compare.py"),
            "--image-path", str(image_path),
            "--npz-path", str(npz_path),
            "--output-dir", str(tmp_fig_dir),
            "--max-tokens", "8",
            "--alpha", "0.45",
        ]

        print(f"\n=== Plot {idx} / {model_tag} ===")
        run_command(cmd, cwd=ROOT)

        generated_fig = tmp_fig_dir / "single_image_heatmap_grid.png"
        assert generated_fig.exists(), f"No se generó figura: {generated_fig}"

        shutil.copy2(generated_fig, explanation_path)
    else:
        print(f"Figura existente, salteando: {idx} / {model_tag}")

    figure_records.append({
        **rec,
        "original_path": str(original_path.relative_to(ROOT)),
        "explanation_path": str(explanation_path.relative_to(ROOT)),
    })

figure_records


## 7. Reorganizar arrays crudos

El script genera un único:

```text
compare_heatmaps.npz
```

Esta celda separa ese archivo en NPZ por método, siguiendo el layout final:

```text
outputs/notebook_comparativo/arrays/<idx>__<model_tag>__<method>.npz
```


In [ ]:
METHODS = ["post_softmax", "qk_logits", "gradcam"]

array_records = []

for rec in figure_records:
    idx = rec["idx"]
    model_tag = rec["model_tag"]
    npz_path = ROOT / rec["npz_path"]

    z = np.load(npz_path, allow_pickle=True)

    available_methods = []

    for method in METHODS:
        words_key = f"{method}_words"
        heatmaps_key = f"{method}_heatmaps"
        caption_key = f"{method}_caption"

        if words_key not in z.files or heatmaps_key not in z.files:
            continue

        out_npz = DIRS["arrays"] / f"{idx}__{model_tag}__{method}.npz"

        np.savez_compressed(
            out_npz,
            words=z[words_key],
            heatmaps=z[heatmaps_key],
            caption=z[caption_key] if caption_key in z.files else np.array([""], dtype=object),
        )

        available_methods.append(method)

        array_records.append({
            "idx": idx,
            "model_tag": model_tag,
            "method": method,
            "arrays_path": str(out_npz.relative_to(ROOT)),
            "n_tokens_relevant": int(len(z[words_key])),
            "heatmaps_shape": tuple(z[heatmaps_key].shape),
        })

    print(idx, model_tag, "methods:", available_methods)

pd.DataFrame(array_records)


## 8. Construir cache de captions

Guarda un JSON agregado en:

```text
outputs/notebook_comparativo/captions/captions_bestof3.json
```

En esta v0 usamos una sola seed, pero el formato queda preparado para best-of-3.


In [ ]:
captions_cache = {}

for rec in figure_records:
    idx = rec["idx"]
    model_tag = rec["model_tag"]
    caption_path = ROOT / rec["caption_path"]

    with open(caption_path, "r", encoding="utf-8") as f:
        cap = json.load(f)

    captions_cache.setdefault(idx, {})[model_tag] = cap

captions_cache_path = DIRS["captions"] / "captions_bestof3.json"
with open(captions_cache_path, "w", encoding="utf-8") as f:
    json.dump(captions_cache, f, indent=2, ensure_ascii=False)

print("Guardado:", captions_cache_path.relative_to(ROOT))
captions_cache


## 9. Consolidar métricas espaciales

Une los CSV de cada corrida y agrega columnas `idx` y `model_tag`.

Salida:

```text
outputs/notebook_comparativo/metrics/spatial_per_token.csv
outputs/notebook_comparativo/metrics/spatial_summary.csv
```


In [ ]:
metric_frames = []

for rec in figure_records:
    metrics_csv = ROOT / rec["metrics_csv"]
    df = pd.read_csv(metrics_csv)
    df.insert(0, "model_tag", rec["model_tag"])
    df.insert(0, "idx", rec["idx"])
    metric_frames.append(df)

if metric_frames:
    spatial_per_token = pd.concat(metric_frames, ignore_index=True)
else:
    spatial_per_token = pd.DataFrame()

spatial_per_token_path = DIRS["metrics"] / "spatial_per_token.csv"
spatial_per_token.to_csv(spatial_per_token_path, index=False)

if not spatial_per_token.empty:
    spatial_summary = (
        spatial_per_token
        .groupby(["idx", "model_tag", "method_a", "method_b"], as_index=False)
        .agg(
            n_tokens=("word", "count"),
            pearson_mean=("pearson", "mean"),
            cosine_mean=("cosine", "mean"),
            mse_mean=("mse", "mean"),
            top10_iou_mean=("top10_iou", "mean"),
        )
    )
else:
    spatial_summary = pd.DataFrame()

spatial_summary_path = DIRS["metrics"] / "spatial_summary.csv"
spatial_summary.to_csv(spatial_summary_path, index=False)

print("Guardado:", spatial_per_token_path.relative_to(ROOT))
print("Guardado:", spatial_summary_path.relative_to(ROOT))

display(spatial_per_token)
display(spatial_summary)


## 10. Construir `summary.csv`

Un renglón por `(idx, model_tag)`.

Este CSV es el punto de entrada para inspeccionar resultados después.


In [ ]:
summary_rows = []

for rec in figure_records:
    idx = rec["idx"]
    model_tag = rec["model_tag"]
    caption_path = ROOT / rec["caption_path"]

    with open(caption_path, "r", encoding="utf-8") as f:
        cap = json.load(f)

    n_relevant = 0
    for ar in array_records:
        if ar["idx"] == idx and ar["model_tag"] == model_tag and ar["method"] == "post_softmax":
            n_relevant = ar["n_tokens_relevant"]

    summary_rows.append({
        "idx": idx,
        "model_tag": model_tag,
        "model_path": rec["model_path"],
        "image_path": rec["image_path"],
        "reference": next(item["reference"] for item in ITEMS if item["idx"] == idx),
        "caption": cap.get("caption", ""),
        "chosen_seed": cap.get("chosen_seed", None),
        "n_candidates": len(cap.get("candidates", [])) if isinstance(cap.get("candidates", []), list) else None,
        "n_tokens_total": len(cap.get("token_ids", [])),
        "n_tokens_relevant": n_relevant,
        "explanation_path": rec["explanation_path"],
        "arrays_prefix": str((DIRS["arrays"] / f"{idx}__{model_tag}").relative_to(ROOT)),
        "status": "ok",
        "error": "",
        "created_at": datetime.now().isoformat(timespec="seconds"),
    })

summary = pd.DataFrame(summary_rows)
summary_path = OUT / "summary.csv"
summary.to_csv(summary_path, index=False)

print("Guardado:", summary_path.relative_to(ROOT))
summary


## 11. Mostrar figura principal

En el notebook final esta sección se reemplaza por una selección de ejemplos para el informe.


In [ ]:
for rec in figure_records:
    fig_path = ROOT / rec["explanation_path"]
    display(Markdown(f"## {rec['idx']} / {rec['model_tag']}"))
    display(Markdown(f"`{fig_path.relative_to(ROOT)}`"))
    fig = Image.open(fig_path)
    print("figure size:", fig.size)
    display(fig)


## 12. Checklist para escalar a notebook final

Cuando estén disponibles el dataset y los checkpoints, cambiar:

```python
RUN_MODE = "visual_test_indices"

MODELS = {
    "base": ROOT / "models/blip_base",
    "ft5k": ROOT / "models/blip_finetuned_5k/best",
    "ft10k": ROOT / "models/blip_finetuned_10k/best",
}
```

Y construir `ITEMS` desde:

```text
data/visual_test_indices.json
```

usando `load_mimic_dataset`.

Pendientes reales para la versión final:

```text
1. Recuperar/cachear dataset MIMIC-CXR.
2. Conseguir checkpoints ft5k y ft10k.
3. Pasar DEVICE a "cuda" en VM/GPU.
4. Cambiar SEEDS a [42, 43, 44].
5. Iterar sobre las 25 radiografías.
```
